# Ejecutor de dbt — Capa Silver y Gold

Este notebook ejecuta el proyecto **dbt** que vive en la carpeta `dbt/` del repo,
directamente desde Databricks (sin necesidad de instalar dbt en tu PC local).

## Qué hace este notebook

1. Instala `dbt-core` y `dbt-databricks` en el cluster.
2. Genera dinámicamente el archivo `profiles.yml` con las credenciales del cluster actual.
3. Ejecuta `dbt build` (corre Silver → Gold + tests).
4. Muestra el resumen de la ejecución.

## Cómo se conecta dbt con Databricks desde aquí

A diferencia de ejecutar dbt en tu PC (que requiere host + http_path + token),
cuando dbt corre **dentro** de Databricks usa las credenciales del cluster
automáticamente vía el token implícito del notebook.

## Para la demo en vivo

Simplemente: **Run all** en este notebook. dbt construye Silver y Gold en vivo,
se ven los logs en tiempo real, y al final las tablas `silver.silver_*` y
`gold.gold_*` quedan reconstruidas por dbt.

## 1. Configuración — variables del proyecto

In [ ]:
import os

# Auto-detectar el catálogo actual
CATALOG_NAME = spark.sql("SELECT current_catalog()").collect()[0][0]

# Configuración del proyecto dbt
DBT_PROJECT_DIR = "/Workspace/Users/orojanom@unicesar.edu.co/wanderbricks-lakehouse-project/dbt"
DBT_PROFILES_DIR = "/tmp/dbt_profiles"

# ============================================================
# Credenciales del workspace
# ============================================================
# Tu Databricks host (de la URL de tu workspace, sin https://)
DATABRICKS_HOST = "dbc-ee3952e5-c721.cloud.databricks.com"

# HTTP Path de tu SQL Warehouse
WAREHOUSE_HTTP_PATH = "/sql/1.0/warehouses/e1291d4c11d7c700"

# ============================================================
# ⚠️ PEGAR AQUÍ TU PERSONAL ACCESS TOKEN (PAT) ⚠️
# ============================================================
# En Databricks Serverless no podemos obtener el token automáticamente,
# por restricciones de seguridad. Hay que generar uno manualmente:
#
#   1. Esquina superior derecha → Settings → Developer
#   2. Access tokens → Generate new token
#   3. Copiar y pegarlo aquí entre comillas
#
# ⚠️ NO subir este archivo al repo con el token real expuesto.
DATABRICKS_TOKEN = "dapi_TU_TOKEN_AQUI"

# ============================================================
# CRÍTICO — Exportar variables de Python al shell
# Sin esto, las celdas %sh no pueden ver $DBT_PROJECT_DIR
# ============================================================
os.environ["DBT_PROJECT_DIR"] = DBT_PROJECT_DIR
os.environ["DBT_PROFILES_DIR"] = DBT_PROFILES_DIR
os.environ["CATALOG_NAME"] = CATALOG_NAME

print(f"Catálogo detectado: {CATALOG_NAME}")
print(f"Proyecto dbt:       {DBT_PROJECT_DIR}")
print(f"Profiles dir:       {DBT_PROFILES_DIR}")
print(f"Host:               {DATABRICKS_HOST}")
print(f"SQL Warehouse:      {WAREHOUSE_HTTP_PATH}")
print(f"Token:              {'✓ configurado' if DATABRICKS_TOKEN != 'dapi_TU_TOKEN_AQUI' else '✗ FALTA pegar tu PAT'}")
print()
print("✓ Variables exportadas al shell (las celdas %sh ahora las verán)")

## 2. Instalar dbt-core + dbt-databricks en el cluster

Esto solo tarda 30-60 segundos la primera vez. Después de instalado, las próximas ejecuciones lo saltean.

In [ ]:
%pip install dbt-core dbt-databricks --quiet

In [ ]:
dbutils.library.restartPython()

## 3. Verificar que dbt está instalado

In [ ]:
%sh dbt --version

## 4. Generar `profiles.yml` dinámicamente

El `profiles.yml` se crea automáticamente con las credenciales del cluster.

**Reemplaza `<TU_WAREHOUSE_ID>` en la celda 1** con el ID de tu SQL Warehouse:

1. En Databricks → SQL Warehouses → tu warehouse activo.
2. Connection details → copia el HTTP Path (algo como `/sql/1.0/warehouses/abc123def456`).
3. Pégalo en la variable `WAREHOUSE_HTTP_PATH` de la celda 1.

In [ ]:
import os

# ============================================================
# Redefinir variables — necesario porque dbutils.library.restartPython()
# en la celda anterior reinició el kernel y borró las variables previas
# ============================================================
CATALOG_NAME = spark.sql("SELECT current_catalog()").collect()[0][0]
DBT_PROJECT_DIR = "/Workspace/Users/orojanom@unicesar.edu.co/wanderbricks-lakehouse-project/dbt"
DBT_PROFILES_DIR = "/tmp/dbt_profiles"

# Credenciales del workspace
DATABRICKS_HOST = "dbc-ee3952e5-c721.cloud.databricks.com"
WAREHOUSE_HTTP_PATH = "/sql/1.0/warehouses/e1291d4c11d7c700"

# ⚠️ PEGAR AQUÍ TU PAT (el mismo que pusiste en la celda 1)
DATABRICKS_TOKEN = "dapi_TU_TOKEN_AQUI"

# Exportar al shell
os.environ["DBT_PROJECT_DIR"] = DBT_PROJECT_DIR
os.environ["DBT_PROFILES_DIR"] = DBT_PROFILES_DIR
os.environ["CATALOG_NAME"] = CATALOG_NAME

# Validar que el token fue configurado
if DATABRICKS_TOKEN == "dapi_TU_TOKEN_AQUI":
    raise ValueError(
        "⚠️ FALTA configurar el token. "
        "Genera un PAT en Databricks (Settings → Developer → Access tokens) "
        "y reemplaza 'dapi_TU_TOKEN_AQUI' en esta celda Y en la celda 1."
    )

# Crear directorio de profiles
os.makedirs(DBT_PROFILES_DIR, exist_ok=True)

# Generar el profiles.yml dinámicamente
profiles_content = f"""
wanderbricks:
  target: dev
  outputs:
    dev:
      type: databricks
      catalog: {CATALOG_NAME}
      schema: silver
      host: {DATABRICKS_HOST}
      http_path: {WAREHOUSE_HTTP_PATH}
      token: {DATABRICKS_TOKEN}
      threads: 4
"""

profiles_path = f"{DBT_PROFILES_DIR}/profiles.yml"
with open(profiles_path, 'w') as f:
    f.write(profiles_content)

print(f"profiles.yml creado en: {profiles_path}")
print("Configuración:")
print(f"  catalog:   {CATALOG_NAME}")
print(f"  host:      {DATABRICKS_HOST}")
print(f"  http_path: {WAREHOUSE_HTTP_PATH}")
print(f"  token:     ***{DATABRICKS_TOKEN[-4:]} (últimos 4 chars)")
print()
print("✓ profiles.yml generado correctamente")

## 5. Verificar conexión — `dbt debug`

Esto prueba que dbt puede hablar con tu Databricks.

In [ ]:
%sh
cd $DBT_PROJECT_DIR && dbt debug --profiles-dir $DBT_PROFILES_DIR

## 6. Ejecutar la capa Silver con dbt

Construye las 6 tablas `silver.silver_*` desde Bronze.

In [ ]:
%sh
cd $DBT_PROJECT_DIR && dbt run --select silver --profiles-dir $DBT_PROFILES_DIR

## 7. Ejecutar la capa Gold con dbt

Construye las 5 tablas Gold (1 fact + 4 dims) desde Silver.

In [ ]:
%sh
cd $DBT_PROJECT_DIR && dbt run --select gold --profiles-dir $DBT_PROFILES_DIR

## 8. Ejecutar tests de calidad de datos

Los tests `not_null`, `unique`, `accepted_values` y `relationships` definidos en `schema.yml`.

In [ ]:
%sh
cd $DBT_PROJECT_DIR && dbt test --profiles-dir $DBT_PROFILES_DIR

## 9. Generar documentación con linaje

Después de esto, el linaje queda visible para `dbt docs serve` (si se ejecuta localmente).

In [ ]:
%sh
cd $DBT_PROJECT_DIR && dbt docs generate --profiles-dir $DBT_PROFILES_DIR

## 10. Validación — verificar tablas creadas por dbt

In [ ]:
%sql
SHOW TABLES IN silver;

In [ ]:
%sql
SHOW TABLES IN gold;

In [ ]:
%sql
-- Conteo de todas las tablas Silver y Gold creadas por dbt
SELECT 'silver_users'        AS tabla, COUNT(*) AS registros FROM silver.silver_users
UNION ALL
SELECT 'silver_destinations' AS tabla, COUNT(*) AS registros FROM silver.silver_destinations
UNION ALL
SELECT 'silver_properties'   AS tabla, COUNT(*) AS registros FROM silver.silver_properties
UNION ALL
SELECT 'silver_bookings'     AS tabla, COUNT(*) AS registros FROM silver.silver_bookings
UNION ALL
SELECT 'silver_payments'     AS tabla, COUNT(*) AS registros FROM silver.silver_payments
UNION ALL
SELECT 'silver_reviews'      AS tabla, COUNT(*) AS registros FROM silver.silver_reviews
UNION ALL
SELECT 'gold_fact_reservas'  AS tabla, COUNT(*) AS registros FROM gold.gold_fact_reservas
UNION ALL
SELECT 'gold_dim_users'      AS tabla, COUNT(*) AS registros FROM gold.gold_dim_users
UNION ALL
SELECT 'gold_dim_properties' AS tabla, COUNT(*) AS registros FROM gold.gold_dim_properties
UNION ALL
SELECT 'gold_dim_destinations' AS tabla, COUNT(*) AS registros FROM gold.gold_dim_destinations
UNION ALL
SELECT 'gold_dim_time'       AS tabla, COUNT(*) AS registros FROM gold.gold_dim_time
ORDER BY tabla;

## Conclusión

Ejecutaste el flujo completo:

1. ✅ dbt instalado en el cluster
2. ✅ Conexión a Databricks verificada
3. ✅ Silver construido por dbt (6 tablas)
4. ✅ Gold construido por dbt (5 tablas)
5. ✅ Tests de calidad ejecutados
6. ✅ Documentación generada

### Para la sustentación

Este notebook reemplaza la ejecución de dbt desde la PC. Demostrar dbt en vivo
se vuelve trivial: abres este notebook → **Run all** → muestras los logs de
`dbt run` y `dbt test` ejecutándose, y al final el conteo de tablas creadas.

### Decisiones de diseño defendibles

- **¿Por qué ejecutar dbt dentro de Databricks?** Las empresas usan Databricks Jobs/Workflows con dbt task para orquestar pipelines productivos. Ejecutar dbt desde un notebook simula ese patrón sin la complejidad del setup de Workflows.
- **¿Por qué `profiles.yml` se genera dinámicamente?** Para evitar hardcodear credenciales en el repo. El token se obtiene del contexto del notebook, que tiene acceso automático al workspace.
- **¿Por qué `%sh` y no Python?** dbt es una CLI — la forma idiomática de invocarlo es vía shell. Databricks soporta `%sh` para ejecutar comandos en el driver del cluster.